In [0]:
%run ../00-common/01-Environment_Config

In [0]:
%run ../00-common/02-Bronze_Helpers

In [0]:
circuit_file = f"{landing_folder_path}/circuits.csv"
circuit_table_name = f"{catalog_name}.{bronze_schema}.circuits"

races_file = f"{landing_folder_path}/races.csv"
races_table_name = f"{catalog_name}.{bronze_schema}.races"

### Ingest Circuits.csv file

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

In [0]:
circuits_df = spark.read.format('csv')\
                .option('header', 'true')\
                .schema(circuits_schema)\
                .option('mode', 'FAILFAST')\
                .load(circuit_file)

# FAILFAST mode for strict data validation

### Add Metadata columns
- source File
- Ingestion Timestamp

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
circuits_final_df.write.format('delta').mode('overwrite').saveAsTable(circuit_table_name)

## Ingest races.csv file

In [0]:
from pyspark.sql.types import IntegerType, DateType

races_schema = StructType([
    StructField('season', IntegerType()),
    StructField('round', IntegerType()),
    StructField('url', StringType()),
    StructField('raceName', StringType()),
    StructField('date', DateType()),
    StructField('circuitId', StringType())
])

In [0]:
races_df = spark.read.format('csv')\
                .option('header', 'true')\
                .schema(races_schema)\
                .option('mode', 'FAILFAST')\
                .load(races_file)

In [0]:
races_final_df = add_ingestion_metadata(races_df)

In [0]:
races_final_df.write.format('delta').mode('overwrite').saveAsTable(races_table_name)